# 128 — Contratos de roles, capacidades y resultados

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Tres capas**: contrato de **rol** (alcance + autoridad + obligaciones), de
**capacidades** (operaciones con JSON Schema de entrada — la forma de una tool
definition) y de **resultados** (esquema + rangos + semántica + caso de error tipado).

**Regla de frontera**: validar entrada Y salida de cada agente. Entre agentes NO
aplica "be liberal in what you accept": tolerar salidas malformadas propaga
corrupción. Ante violación → reintento con el error como feedback → degradar/escalar;
nunca arreglar en silencio.

**SLA de agente**: latencia, coste, tasa de validez... y calidad como *distribución
muestreada* (LLM-judge/tests), jamás perfección por llamada. Contratos publicados =
base del descubrimiento (Agent Card de A2A) y la negociación (Contract Net, 1980).


## 🧮 Ejemplo de referencia

Contrato del worker: `{agent: const, score: [0,1], finding: string ≥ 1}`.

```text
{"agent": "security", "score": 0.6, "finding": "falta threat model"}  ✓
{"score": 1.4}                → violación de rango
{"agent": "security"}         → falta campo requerido
"El repo se ve bien"          → texto libre: la violación más común con LLM
```

Al cuarto caso no se le aplican regex heroicas: se reintenta adjuntando el error de
validación y se degrada tras k intentos.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("multiagent", seed=128)
show(result)


## Reflexión

1. El laboratorio devuelve siempre resultados válidos porque los workers son deterministas. ¿En qué punto exacto del flujo insertarías el validador si fueran LLM, y qué harías con la evidencia de los intentos fallidos?
2. ¿Por qué "score 1.4 → recortar a 1.0 en silencio" es peor que "score 1.4 → reintento visible", si el segundo cuesta una llamada extra?
3. Diseña una garantía de SLA de *calidad* para el worker documentation que puedas medir de verdad esta semana. ¿Qué muestra, qué juez, qué umbral?
